# M2 emotion training on Colab

Runtime > Change runtime type > T4 GPU. Runs the three M2 configs and saves
runs + reports to Google Drive so nothing is lost when the session ends.

FER2013 comes from Kaggle. You need:
- an API token (kaggle.com > Settings > API Tokens > Generate, starts with `KGAT_`).
  Either add it as a Colab secret named `KAGGLE_API_TOKEN` (key icon in the left
  bar) or paste it when the data cell asks.
- the competition rules accepted at
  https://www.kaggle.com/c/challenges-in-representation-learning-facial-expression-recognition-challenge/rules

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/interviewlens"
!mkdir -p {DRIVE}

In [ ]:
!git clone -q https://github.com/mohanmarimuthu1/Interview_Analyzer.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -e ".[train]"

## Data

Skips the Kaggle step if `fer2013.csv` is already on Drive from an earlier run.

In [ ]:
import os, shutil

os.makedirs("data/raw/fer2013", exist_ok=True)
cached = f"{DRIVE}/fer2013.csv"
if os.path.exists(cached):
    shutil.copy(cached, "data/raw/fer2013/fer2013.csv")
else:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ")
    !pip install -q -U kaggle
    !kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -f fer2013.tar.gz -p data/raw/fer2013
    !cd data/raw/fer2013 && (ls *.zip >/dev/null 2>&1 && unzip -oq *.zip; true) && tar xzf fer2013.tar.gz && mv fer2013/fer2013.csv .
    shutil.copy("data/raw/fer2013/fer2013.csv", cached)

!mkdir -p data/raw/ferplus
!curl -sfL -o data/raw/ferplus/fer2013new.csv https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv
!wc -l data/raw/fer2013/fer2013.csv data/raw/ferplus/fer2013new.csv

## Runs

1. ResNet-18 on FER2013 labels: v1 parity check (target >= 68% test acc)
2. ResNet-18 on FER+ labels
3. ConvNeXt-T on FER+ labels (lower lr, it is a bigger pretrained model)

Each run: best epoch picked on PublicTest, reported on PrivateTest.

In [ ]:
RUN = f"hydra.run.dir={DRIVE}/runs/" + "${run_name}"
REPORTS = f"reports_dir={DRIVE}/reports/emotion"
!python -m interviewlens.train_emotion data.labels=fer2013 model=resnet18 '{RUN}' {REPORTS}

In [ ]:
!python -m interviewlens.train_emotion data.labels=ferplus model=resnet18 '{RUN}' {REPORTS}

In [ ]:
!python -m interviewlens.train_emotion data.labels=ferplus model=convnext_t train.lr=1e-4 train.batch_size=64 '{RUN}' {REPORTS}

## Results

Download `reports/emotion/` from Drive into the repo's `reports/emotion/`
(metrics.json + confusion.png per run). Checkpoints stay in `runs/` on Drive.

In [ ]:
import glob, json
for p in sorted(glob.glob(f"{DRIVE}/reports/emotion/*/metrics.json")):
    m = json.load(open(p))
    print(f"{m['run']:32s} val {m['best_val_acc']:.4f}  test acc {m['test_acc']:.4f}  macro-F1 {m['test_macro_f1']:.4f}")